## Purpose
I want to gather data to answer the central question: What are the most common type of violation and does this change by ZIP code or facility type.
## Outline
* Top Violations codes
* Violations by facility type
* Violations by ZIP code
* Approximate the priority of a violation code by weighing them by the fail share


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data\\clean\\food_inspections_cleaned.csv", dtype={"zip": str})

## Top Violations

In [3]:
top10_overall = df.groupby("violation_code").size().reset_index(name="count").sort_values("count", ascending=False).head(10)
top10_overall_set = set(top10_overall["violation_code"])

## Violations by Facility Type

In [4]:
violations_by_facility_type_df = (df.groupby(["facility_type_clean", "violation_code"])
                                  .size().reset_index(name="count")
                                  .sort_values(["facility_type_clean","count"], ascending=[True, False])
                                  .groupby("facility_type_clean")
                                  .head(10)
)

violations_by_facility_type_df

,facility_type_clean,violation_code,count
53,BAKERY,55.0,1333
36,BAKERY,38.0,1203
32,BAKERY,34.0,1140
31,BAKERY,33.0,1112
33,BAKERY,35.0,1055
...,...,...,...
915,TAVERN,41.0,79
906,TAVERN,32.0,74
895,TAVERN,18.0,73
889,TAVERN,10.0,56


In [5]:
facility_top_violations_dict = violations_by_facility_type_df.groupby("facility_type_clean")["violation_code"].apply(set).to_dict()
facility_top_violations_dict

{'BAKERY': {32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 41.0, 47.0, 55.0},
 'BANQUET': {10.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 55.0, 58.0},
 'BAR': {5.0, 18.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 55.0},
 'CAFE': {10.0, 18.0, 32.0, 33.0, 34.0, 35.0, 38.0, 41.0, 51.0, 55.0},
 'CATERING': {32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 47.0, 51.0, 55.0},
 'COMMUNITY BUILDINGS': {10.0,
  33.0,
  34.0,
  35.0,
  36.0,
  38.0,
  41.0,
  51.0,
  55.0,
  56.0},
 'DAYCARE': {10.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 51.0, 55.0},
 'GAS STATION': {5.0, 10.0, 32.0, 33.0, 34.0, 35.0, 38.0, 41.0, 49.0, 55.0},
 'GROCERY': {32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 47.0, 49.0, 55.0},
 'HEALTHCARE': {10.0, 32.0, 33.0, 34.0, 35.0, 38.0, 47.0, 49.0, 51.0, 55.0},
 'LIQUOR': {5.0, 10.0, 18.0, 32.0, 33.0, 34.0, 35.0, 38.0, 41.0, 55.0},
 'MOBILE FOOD': {2.0, 3.0, 5.0, 18.0, 30.0, 32.0, 33.0, 37.0, 38.0, 55.0},
 'OTHER': {3.0, 10.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 55.0},
 'RESTAURANT':

In [6]:
jaccard_similarity_dict = {}

def jaccard_similarity(set1: set, set2: set) -> float:
    """Jaccard Similarity measures set similarity as the intersection of two sets divided by the union of two sets."""
    return len(set1 & set2) / len(set1 | set2)

for key, value in facility_top_violations_dict.items():
    jaccard_similarity_value = jaccard_similarity(top10_overall_set, value)
    jaccard_similarity_dict[key] = jaccard_similarity_value

jaccard_similarity_dict

{'BAKERY': 0.8181818181818182,
 'BANQUET': 0.6666666666666666,
 'BAR': 0.6666666666666666,
 'CAFE': 0.5384615384615384,
 'CATERING': 0.8181818181818182,
 'COMMUNITY BUILDINGS': 0.5384615384615384,
 'DAYCARE': 0.6666666666666666,
 'GAS STATION': 0.6666666666666666,
 'GROCERY': 1.0,
 'HEALTHCARE': 0.6666666666666666,
 'LIQUOR': 0.5384615384615384,
 'MOBILE FOOD': 0.25,
 'OTHER': 0.6666666666666666,
 'RESTAURANT': 1.0,
 'SCHOOL': 0.6666666666666666,
 'TAVERN': 0.5384615384615384}

In [7]:
sum = 0

for key, value in jaccard_similarity_dict.items():
    sum += value

sum / len(jaccard_similarity_dict)

0.6691797785547785

Grocery stores, and restaurants have the same common violations as the top 10, with catering, and bakeries having a majority of the same most common violations. All of the other facility types hover around 50-60% of violations in common, with Mobile Food having notably novel issues, sharing only 25% of them. With a mean of around 0.67 this means, the issues are mostly shared by facilities.

## Violations by ZIP Code

In [8]:
valid_zips = df.groupby("zip")["inspection_id"].nunique().loc[lambda x: x >= 50].index

In [9]:
violations_by_zip_code_df = (df[df["zip"].isin(valid_zips)]
                                .groupby(["zip", "violation_code"])
                                  .size().reset_index(name="count")
                                  .sort_values(["zip","count"], ascending=[True, False])
                                  .groupby("zip")
                                  .head(10)
)

violations_by_zip_code_df

,zip,violation_code,count
32,60601.0,33.0,1184
33,60601.0,34.0,1082
37,60601.0,38.0,927
31,60601.0,32.0,833
54,60601.0,55.0,826
...,...,...,...
3687,60827.0,33.0,30
3695,60827.0,41.0,30
3706,60827.0,53.0,30
3690,60827.0,36.0,26


In [10]:
zip_top10_dict = violations_by_zip_code_df.groupby("zip")["violation_code"].apply(set).to_dict()
zip_top10_dict

{'60601.0': {10.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 47.0, 55.0},
 '60602.0': {10.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 47.0, 55.0},
 '60603.0': {3.0, 10.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 47.0, 55.0},
 '60604.0': {3.0, 10.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 55.0},
 '60605.0': {10.0, 32.0, 33.0, 34.0, 35.0, 38.0, 41.0, 49.0, 55.0, 58.0},
 '60606.0': {3.0, 10.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 55.0},
 '60607.0': {3.0, 10.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 55.0},
 '60608.0': {3.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 47.0, 55.0},
 '60609.0': {32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 47.0, 51.0, 55.0},
 '60610.0': {3.0, 10.0, 32.0, 33.0, 34.0, 35.0, 38.0, 41.0, 47.0, 55.0},
 '60611.0': {3.0, 10.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 47.0, 55.0},
 '60612.0': {3.0, 32.0, 33.0, 34.0, 35.0, 36.0, 38.0, 41.0, 49.0, 55.0},
 '60613.0': {3.0, 10.0, 32.0, 33.0, 34.0, 35.0, 38.0, 41.0, 47.0, 55.0},
 '60614.0': {32.0, 33.0, 34.0, 35.0, 36.0, 38.0

In [11]:
jaccard_similarity_dict = {}

for key, value in zip_top10_dict.items():
    jaccard_similarity_value = jaccard_similarity(top10_overall_set, value)
    jaccard_similarity_dict[key] = jaccard_similarity_value

jaccard_similarity_dict

{'60601.0': 0.8181818181818182,
 '60602.0': 0.8181818181818182,
 '60603.0': 0.6666666666666666,
 '60604.0': 0.6666666666666666,
 '60605.0': 0.6666666666666666,
 '60606.0': 0.6666666666666666,
 '60607.0': 0.6666666666666666,
 '60608.0': 0.8181818181818182,
 '60609.0': 0.8181818181818182,
 '60610.0': 0.6666666666666666,
 '60611.0': 0.6666666666666666,
 '60612.0': 0.8181818181818182,
 '60613.0': 0.6666666666666666,
 '60614.0': 0.8181818181818182,
 '60615.0': 0.8181818181818182,
 '60616.0': 0.8181818181818182,
 '60617.0': 0.6666666666666666,
 '60618.0': 0.8181818181818182,
 '60619.0': 1.0,
 '60620.0': 0.8181818181818182,
 '60621.0': 0.8181818181818182,
 '60622.0': 0.8181818181818182,
 '60623.0': 1.0,
 '60624.0': 0.8181818181818182,
 '60625.0': 1.0,
 '60626.0': 0.8181818181818182,
 '60628.0': 0.8181818181818182,
 '60629.0': 0.8181818181818182,
 '60630.0': 0.8181818181818182,
 '60631.0': 0.6666666666666666,
 '60632.0': 0.8181818181818182,
 '60633.0': 0.6666666666666666,
 '60634.0': 0.8181818

In [12]:
sum = 0

for key, value in jaccard_similarity_dict.items():
    sum += value

sum / len(jaccard_similarity_dict)

0.8061317213859587

There are some regional differences in the top 10 violations but with a mean of 0.80 this means the regional differences are not as signficant as facility type differences.

In [32]:
df[df["violation_code"] == 16]

,Unnamed: 0,inspection_id,violation_code,violation_text,inspection_date,results,risk,dba_name,facility_type,address,zip,year,month,facility_type_clean
4,4,2638381,16.0,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,06/11/2026,Pass,Risk 1 (High),TAQUERIA LOS GALLOS INC,RESTAURANT,4209-4211 W 26TH ST,60623.0,2026,6,RESTAURANT
7,7,2638318,16.0,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,06/11/2026,Fail,Risk 1 (High),CEVICHERIA RIO BALSAS,RESTAURANT,4545 W DIVISION ST,60651.0,2026,6,RESTAURANT
19,19,2638338,16.0,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,06/11/2026,Fail,Risk 1 (High),BIXI BREWERY,RESTAURANT,2515-2517 N MILWAUKEE AVE,60647.0,2026,6,RESTAURANT
141,141,2638216,16.0,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,06/10/2026,Fail,Risk 1 (High),BROADWAY KOHI,RESTAURANT,2904 N BROADWAY AVE,60657.0,2026,6,RESTAURANT
177,177,2638223,16.0,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,06/10/2026,Pass,Risk 1 (High),LA ESTRELLA,GROCERY STORE,3835 W 26TH ST,60623.0,2026,6,GROCERY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1099200,1099200,118300,16.0,"16. FOOD PROTECTED DURING STORAGE, PREPARATION...",01/06/2010,Fail,Risk 1 (High),SEVEN TREASURES,RESTAURANT,2312 S WENTWORTH AVE,60616.0,2010,1,RESTAURANT
1099216,1099216,154221,16.0,"16. FOOD PROTECTED DURING STORAGE, PREPARATION...",01/06/2010,Fail,Risk 2 (Medium),SUBWAY SANDWICHES AND SALADS,RESTAURANT,1513 W FULLERTON AVE,60614.0,2010,1,RESTAURANT
1099257,1099257,67763,16.0,"16. FOOD PROTECTED DURING STORAGE, PREPARATION...",01/06/2010,Fail,Risk 2 (Medium),SAN MIGUEL BAKERY INC.,BAKERY,1607 W MONTROSE AVE,60613.0,2010,1,BAKERY
1099313,1099313,74261,16.0,"16. FOOD PROTECTED DURING STORAGE, PREPARATION...",01/06/2010,Fail,Risk 2 (Medium),CHICAGO STEAKHOUSE,RESTAURANT,219 E 47TH ST,60653.0,2010,1,RESTAURANT
